In [1]:
!pip install -q nltk

In [2]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [3]:
%%writefile parser.py
import nltk
import sys

TERMINALS = """
Adj -> "country" | "dreadful" | "enigmatical" | "little" | "moist" | "red"
Adv -> "down" | "here" | "never"
Conj -> "and" | "until"
Det -> "a" | "an" | "his" | "my" | "the"
N -> "armchair" | "companion" | "day" | "door" | "hand" | "he" | "himself"
N -> "holmes" | "home" | "i" | "mess" | "paint" | "palm" | "pipe" | "she"
N -> "smile" | "thursday" | "walk" | "we" | "word"
P -> "at" | "before" | "in" | "of" | "on" | "to"
V -> "arrived" | "came" | "chuckled" | "had" | "lit" | "said" | "sat"
V -> "smiled" | "tell" | "were"
"""

NONTERMINALS = """
S -> NP VP | S Conj S
NP -> N | Det N | Det AdjP N | NP PP
AdjP -> Adj | Adj AdjP
VP -> V | V NP | V PP | V Adv | Adv VP | V NP PP | V PP Adv | V NP Adv
PP -> P NP
"""

grammar = nltk.CFG.fromstring(NONTERMINALS + TERMINALS)
parser = nltk.ChartParser(grammar)


def main():

    # If filename specified, read sentence from file
    if len(sys.argv) == 2:
        with open(sys.argv[1]) as f:
            s = f.read()

    # Otherwise, get sentence as input
    else:
        s = input("Sentence: ")

    # Convert input into list of words
    s = preprocess(s)

    # Attempt to parse sentence
    try:
        trees = list(parser.parse(s))
    except ValueError as e:
        print(e)
        return
    if not trees:
        print("Could not parse sentence.")
        return

    # Print each tree with noun phrase chunks
    for tree in trees:
        tree.pretty_print()

        print("Noun Phrase Chunks")
        for np in np_chunk(tree):
            print(" ".join(np.flatten()))


def preprocess(sentence):
    """
    Convert `sentence` to a list of its words.
    Pre-process sentence by converting all characters to lowercase
    and removing any word that does not contain at least one alphabetic
    character.
    """
    tokens = nltk.word_tokenize(sentence.lower())
    words = []

    for token in tokens:
        if any(char.isalpha() for char in token):
            words.append(token)

    return words


def np_chunk(tree):
    """
    Return a list of all noun phrase chunks in the sentence tree.
    A noun phrase chunk is defined as any subtree of the sentence
    whose label is "NP" that does not itself contain any other
    noun phrases as subtrees.
    """
    chunks = []

    for subtree in tree.subtrees(lambda t: t.label() == "NP"):
        has_inner_np = False

        for child in subtree.subtrees():
            if child != subtree and child.label() == "NP":
                has_inner_np = True
                break

        if not has_inner_np:
            chunks.append(subtree)

    return chunks


if __name__ == "__main__":
    main()

Writing parser.py


In [4]:
!grep -n "NotImplementedError" parser.py

In [5]:
!python parser.py 1.txt

        S     
   _____|___   
  NP        VP
  |         |  
  N         V 
  |         |  
holmes     sat

Noun Phrase Chunks
holmes


In [6]:
!python parser.py 5.txt

                    S                             
      ______________|_________                     
     |                        VP                  
     |               _________|_______             
     |              |                 NP          
     |              |      ___________|________    
     NP             |     |          AdjP      |  
  ___|______        |     |           |        |   
Det         N       V    Det         Adj       N  
 |          |       |     |           |        |   
 my     companion smiled  an     enigmatical smile

Noun Phrase Chunks
my companion
an enigmatical smile
